In [1]:
import os
import re
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm

# Path where your CSVs live
data_path = "./"  # adjust if needed

# All model files you want to include
csv_files = [
    "aya-expanse-32b_test.csv",
    "gemma3-12b_test.csv",
    "gemma3-27b_test.csv",
    "gpt4o_test.csv",
    "llama3.1_test.csv",
    "llama3.2-latest_test.csv",
    "llama3.3-latest_test.csv",
    "mistral-small-3.2-24b_test.csv",
    "mistral-small3.1-latest_test.csv",
    "qwen2.5-14b_test.csv",
    "qwen2.5-32b_test.csv",
    "qwen2.5-72b_test.csv"
    # 🔴 Excluded qwen3
]

# Collect all results
all_results = []

for file in csv_files:
    path = os.path.join(data_path, file)
    if not os.path.exists(path):
        print(f"⚠️ File not found: {file}")
        continue
    
    df = pd.read_csv(path)
    model_name = file.replace("_test.csv", "")
    
    # Extract the first letter A/B/C from 'answer'
    df["choice"] = df["answer"].astype(str).str.extract(r"\b([ABC])\b")
    df["model"] = model_name
    
    all_results.append(df[["scenario_code", "model", "choice"]])

# Combine into one dataframe
all_df = pd.concat(all_results, ignore_index=True)

# Compute distribution (counts)
distribution = all_df.groupby(["model", "choice"]).size().unstack(fill_value=0)

# Normalize to proportions
distribution_prop = distribution.div(distribution.sum(axis=1), axis=0)

# Save results to file
output_file = "decision_comparison.csv"
with pd.ExcelWriter(output_file.replace(".csv", ".xlsx")) as writer:
    distribution.to_excel(writer, sheet_name="Counts")
    distribution_prop.to_excel(writer, sheet_name="Proportions")
    all_df.to_excel(writer, sheet_name="All_Decisions", index=False)

# Also save CSV (counts + proportions)
distribution.to_csv("decision_comparison_counts.csv")
distribution_prop.to_csv("decision_comparison_proportions.csv")
all_df.to_csv("decision_comparison_all.csv", index=False)

print(f"\n✅ Results saved as:")
print(f"- decision_comparison.xlsx (counts, proportions, raw decisions)")
print(f"- decision_comparison_counts.csv")
print(f"- decision_comparison_proportions.csv")
print(f"- decision_comparison_all.csv")

# --- Define shades of blue for A, B, C ---
# pick 3 different levels from the "Blues" colormap
blue_colors = [cm.Blues(0.4), cm.Blues(0.6), cm.Blues(0.8)]
# (lighter → darker, for A→B→C)

# --- Plot absolute counts ---
ax1 = distribution.plot(
    kind="bar", stacked=True, figsize=(12,6),
    color=blue_colors
)
plt.title("Decision Distribution (Counts)")
plt.ylabel("Count")
plt.xlabel("Model")
plt.xticks(rotation=45, ha="right")
plt.legend(title="Choice", labels=["A","B","C"])
plt.tight_layout()
plt.savefig("decision_distribution_counts.pdf")
plt.close()

# --- Plot proportions ---
ax2 = distribution_prop.plot(
    kind="bar", stacked=True, figsize=(12,6),
    color=blue_colors
)
plt.title("Decision Distribution (Proportions)")
plt.ylabel("Proportion")
plt.xlabel("Model")
plt.xticks(rotation=45, ha="right")
plt.legend(title="Choice", labels=["A","B","C"])
plt.tight_layout()
plt.savefig("decision_distribution_proportions.pdf")
plt.close()



✅ Results saved as:
- decision_comparison.xlsx (counts, proportions, raw decisions)
- decision_comparison_counts.csv
- decision_comparison_proportions.csv
- decision_comparison_all.csv
